In [5]:
import operator
from datetime import datetime
from typing import Annotated, TypedDict

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# --- 1. TOOLS DEFINITION ---
@tool
def get_current_location():
    """Get the user's current city."""
    return "Hong Kong"

@tool
def get_current_date():
    """Get today's date in YYYY-MM-DD format."""
    return f"Today is {datetime.now().strftime('%Y-%m-%d')}."

@tool
def get_historical_weather(location: str, date: str):
    """Get weather for a specific city and date."""
    return f"The weather in {location} on {date} was Rainy, 19°C."

TOOLS = [get_current_location, get_current_date, get_historical_weather]

# --- 2. IMPROVED REACT PROMPT ---
# We tell the model to use the THOUGHT/ACTION cycle, but strictly use tool calls for Action.
REACT_SYSTEM_PROMPT = """You are a ReAct assistant. 
For every user request, you MUST:
1. Thought: Reason about what tools to use.
2. Tool Call: Call the necessary tools to get info.
3. Final Answer: Answer only after you have all info from observations.

IMPORTANT: To determine "yesterday", you MUST call get_current_date first.
To find the location, you MUST call get_current_location.
Do not guess information."""

class State(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def call_model(state: State):
    # Ensure the system prompt is always present for every reasoning step
    messages = state["messages"]
    if not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=REACT_SYSTEM_PROMPT)] + messages
    
    response = model.invoke(messages)
    return {"messages": [response]}

# Initialize Model
model = ChatOllama(model="qwen2.5:7b", temperature=0).bind_tools(TOOLS)

# Build Graph
workflow = StateGraph(State)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(TOOLS))
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", lambda x: "tools" if x["messages"][-1].tool_calls else END)
workflow.add_edge("tools", "agent")

app = workflow.compile(checkpointer=MemorySaver())

# --- 3. EXECUTION ---
def ask_agent(question: str, thread_id: str = "fixed_session"):
    config = {"configurable": {"thread_id": thread_id}}
    inputs = {"messages": [HumanMessage(content=question)]}
    
    print(f"--- Starting Task ---\n")
    for event in app.stream(inputs, config, stream_mode="values"):
        msg = event["messages"][-1]
        msg.pretty_print()

if __name__ == "__main__":
    ask_agent("我在這裡昨天天氣如何？請展示你的推理步驟。")

--- Starting Task ---

================================ Human Message =================================

我在這裡昨天天氣如何？請展示你的推理步驟。
================================== Ai Message ==================================

好的，我將按照以下步驟來回答您的問題：

1. 使用 `get_current_location` 函數獲取當前位置。
2. 使用 `get_current_date` 函數獲取今天的日期。
3. 根據今天的日期計算出昨天的日期。
4. 使用 `get_historical_weather` 函數獲取昨天該地點的天氣情況。

現在，我將開始執行這些步驟。
Tool Calls:
  get_current_location (ad04fccf-695b-4620-87aa-2c15f316df59)
 Call ID: ad04fccf-695b-4620-87aa-2c15f316df59
  Args:
================================= Tool Message =================================
Name: get_current_location

Hong Kong
================================== Ai Message ==================================
Tool Calls:
  get_current_date (3c469a10-d5bc-4004-b0b4-5c1fae872ccf)
 Call ID: 3c469a10-d5bc-4004-b0b4-5c1fae872ccf
  Args:
================================= Tool Message =================================
Name: get_current_date

Today is 2026-03-29.
================================